# c.1330 — XRP Decision Transformer : validation fold-wise (§C doctrine)

**Question** : le **Decision Transformer (DT)** déployé en fold-wise (entraînement
par anchor trimestriel, validation sur le trimestre suivant) bat-il **statistiquement**
le buy-and-hold (BH) et/ou le **momentum naked** (baseline directionnelle triviale)
sur XRP-USD ?

**Gap comblé** : l'Epic **#1454** (fold-wise deployment protocol) a livré le driver
de sweep multi-anchor + multi-seed. Ce notebook ingère le JSON de sweep et applique
la **doctrine §C** (BEATS = edge_pp > 0 AND edge_σ ≥ +2.0 AND dm_p_median < 0.05)
pour trancher honnêtement entre BEATS / NO-BEATS / INCONCLUSIVE.

**Sources** :
- Sweep JSON : `results/xrp_dt_validation/foldwise_20260806_165146.json` (gitignored,
  produit par le runner GPU po-2024 — sweep 47.7 min sur RTX 3070).
- Snapshot embarqué (fallback) : cellule code §0 ci-dessous (les 4 buckets summary
  + decay + table per_seed agrégée).
- Driver Epic #1454 : PR de déploiement du protocole (MERGED antérieurement).
- §C doctrine : walk-forward + ≥4 seeds parmi {0,1,7,42,99} + Diebold-Mariano HAC
  Newey-West (Ledoit-Wolf 2008) + verdict honnête.

**Verdict global (résumé)** : **NO-BEATS × 4 buckets**. DT ne bat BH nulle part avec
edge_σ ≥ 2 (max observé : +0.76σ, fresh/sliding). DT perd contre momentum naked
sur les 4 buckets (-0.19 à -0.39 Sharpe). Signal de dégénérescence détecté :
sur l'anchor 2025-Q4 en mode `fresh`, les 4 seeds convergent vers
`dt_net ≈ -bh_sharpe` au bit près — modèle dégénéré qui imite l'inverse de BH.

**Recommandation opérationnelle** : ne pas activer le fold-wise deployment sur XRP.
Le coût de ré-entraînement (50-135s/anchor selon window) ne se justifie pas vu
l'edge négatif contre momentum naked.

In [1]:
# Section 0 — Chargement du sweep JSON
#
# Strategie en 2 etapes :
# 1. Essayer de charger le JSON frais depuis le dossier gitignored du repo
#    (chemin relatif worktree-agnostic via Path(__file__) ne fonctionne pas
#    en Papermill kernel -- on utilise donc un chemin relatif au CWD).
# 2. Si le JSON est absent (CI / autre machine), retomber sur le SNAPSHOT
#    embarque ci-dessous : summary_by_bucket + decay + table per_seed
#    agregee. Ce snapshot est un sous-ensemble fidele du JSON complet.

import json
from pathlib import Path

CANDIDATE_PATHS = [
    Path('results/xrp_dt_validation/foldwise_20260806_165146.json'),
    Path('../results/xrp_dt_validation/foldwise_20260806_165146.json'),
]

loaded = None
for p in CANDIDATE_PATHS:
    if p.exists():
        with open(p, encoding='utf-8') as f:
            loaded = json.load(f)
        print(f'[load] OK from {p}')
        break

if loaded is None:
    # SNAPSHOT EMBARQUE (fallback) -- reproduce fidelement les champs cles
    # du JSON foldwise_20260806_165146.json (32 modeles, 4 buckets, sweep
    # 47.7 min GPU po-2024).
    loaded = {
        'timestamp': '20260806_165146',
        'coin': 'XRP-USD',
        'device': 'cuda',
        'smoke': False,
        'data_hash': '6e49685b9f9ab297',
        'sweep_elapsed_s': 2862.5,
        'config': {
            'anchors': ['2025-09-30', '2025-12-31', '2026-03-31', '2026-06-30'],
            'seeds': [0, 1, 7, 42],
            'modes': ['fresh', 'aged-1q'],
            'windows': ['sliding', 'expanding'],
            'holdout_days': 90,
            'gap_days': 10,
            'commission_bps': 10,
        },
        'summary_by_bucket': [
            {'label': 'fresh/sliding', 'n_models': 8,
             'dt_net_sharpe_mean': 0.5859, 'dt_net_sharpe_std': 1.3646,
             'bh_sharpe_mean': -0.8992, 'edge_mean_pp': 148.51,
             'edge_sigma': 0.76, 'dm_p_median': 0.309, 'mean_retrain_s': 49.8},
            {'label': 'aged-1q/sliding', 'n_models': 8,
             'dt_net_sharpe_mean': 0.5145, 'dt_net_sharpe_std': 1.4031,
             'bh_sharpe_mean': -0.8992, 'edge_mean_pp': 141.37,
             'edge_sigma': 0.71, 'dm_p_median': 0.32725, 'mean_retrain_s': 47.7},
            {'label': 'fresh/expanding', 'n_models': 8,
             'dt_net_sharpe_mean': 0.3894, 'dt_net_sharpe_std': 1.2822,
             'bh_sharpe_mean': -0.8992, 'edge_mean_pp': 128.86,
             'edge_sigma': 0.65, 'dm_p_median': 0.309, 'mean_retrain_s': 135.8},
            {'label': 'aged-1q/expanding', 'n_models': 8,
             'dt_net_sharpe_mean': 0.4809, 'dt_net_sharpe_std': 1.1568,
             'bh_sharpe_mean': -0.8992, 'edge_mean_pp': 138.01,
             'edge_sigma': 0.74, 'dm_p_median': 0.309, 'mean_retrain_s': 124.5},
        ],
        'decay_fresh_vs_aged1q': {
            'sliding': {'fresh_edge_pp': 148.51, 'aged1q_edge_pp': 141.37,
                        'decay_pp_per_quarter': 7.14},
            'expanding': {'fresh_edge_pp': 128.86, 'aged1q_edge_pp': 138.01,
                          'decay_pp_per_quarter': -9.15},
        },
        'per_seed_snapshot': [
            # fresh/sliding -- 4 seeds x 2 anchors valides (2025-Q3 + 2025-Q4)
            {'bucket': 'fresh/sliding', 'seed': 0, 'train_end': '2025-09-30',
             'dt_net_sharpe': -2.1876, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.0384},
            {'bucket': 'fresh/sliding', 'seed': 1, 'train_end': '2025-09-30',
             'dt_net_sharpe': 0.3538, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.3524},
            {'bucket': 'fresh/sliding', 'seed': 7, 'train_end': '2025-09-30',
             'dt_net_sharpe': 0.8853, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.5377},
            {'bucket': 'fresh/sliding', 'seed': 42, 'train_end': '2025-09-30',
             'dt_net_sharpe': -0.6012, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.1563},
            # SUSPECT cross-seed dup : 4 seeds x 2025-12-31 convergent au bit
            {'bucket': 'fresh/sliding', 'seed': 0, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'fresh/sliding', 'seed': 1, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'fresh/sliding', 'seed': 7, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'fresh/sliding', 'seed': 42, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            # aged-1q/sliding -- 8 entrees (2 anchors x 4 seeds)
            {'bucket': 'aged-1q/sliding', 'seed': 0, 'train_end': '2025-06-30',
             'dt_net_sharpe': -1.9027, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.0289},
            {'bucket': 'aged-1q/sliding', 'seed': 1, 'train_end': '2025-06-30',
             'dt_net_sharpe': -0.3292, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.1553},
            {'bucket': 'aged-1q/sliding', 'seed': 7, 'train_end': '2025-06-30',
             'dt_net_sharpe': 1.3049, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.5362},
            {'bucket': 'aged-1q/sliding', 'seed': 42, 'train_end': '2025-06-30',
             'dt_net_sharpe': -1.0652, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.3801},
            {'bucket': 'aged-1q/sliding', 'seed': 0, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'aged-1q/sliding', 'seed': 1, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5455, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.3236},
            {'bucket': 'aged-1q/sliding', 'seed': 7, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.4826, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.3309},
            {'bucket': 'aged-1q/sliding', 'seed': 42, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5211, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.955},
            # fresh/expanding -- 8 entrees
            {'bucket': 'fresh/expanding', 'seed': 0, 'train_end': '2025-09-30',
             'dt_net_sharpe': -1.2701, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.1004},
            {'bucket': 'fresh/expanding', 'seed': 1, 'train_end': '2025-09-30',
             'dt_net_sharpe': -0.6036, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.8791},
            {'bucket': 'fresh/expanding', 'seed': 7, 'train_end': '2025-09-30',
             'dt_net_sharpe': -0.2784, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.2299},
            {'bucket': 'fresh/expanding', 'seed': 42, 'train_end': '2025-09-30',
             'dt_net_sharpe': -0.97, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.0533},
            {'bucket': 'fresh/expanding', 'seed': 0, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'fresh/expanding', 'seed': 1, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'fresh/expanding', 'seed': 7, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'fresh/expanding', 'seed': 42, 'train_end': '2025-12-31',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            # aged-1q/expanding -- 8 entrees
            {'bucket': 'aged-1q/expanding', 'seed': 0, 'train_end': '2025-06-30',
             'dt_net_sharpe': -0.4017, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.1411},
            {'bucket': 'aged-1q/expanding', 'seed': 1, 'train_end': '2025-06-30',
             'dt_net_sharpe': -0.7489, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.432},
            {'bucket': 'aged-1q/expanding', 'seed': 7, 'train_end': '2025-06-30',
             'dt_net_sharpe': -0.6409, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.2337},
            {'bucket': 'aged-1q/expanding', 'seed': 42, 'train_end': '2025-06-30',
             'dt_net_sharpe': -0.5984, 'momentum_naked_net_sharpe': 1.3084,
             'bh_sharpe': -0.2348, 'dm_p': 0.1094},
            {'bucket': 'aged-1q/expanding', 'seed': 0, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'aged-1q/expanding', 'seed': 1, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'aged-1q/expanding', 'seed': 7, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
            {'bucket': 'aged-1q/expanding', 'seed': 42, 'train_end': '2025-09-30',
             'dt_net_sharpe': 1.5593, 'momentum_naked_net_sharpe': 0.2449,
             'bh_sharpe': -1.5636, 'dm_p': 0.309},
        ],
    }
    print('[load] FALLBACK to embedded snapshot (JSON absent on disk)')

d = loaded
print()
print(f'timestamp: {d["timestamp"]}  coin: {d["coin"]}  device: {d["device"]}')
print(f'data_hash: {d["data_hash"]}  sweep_elapsed_s: {d["sweep_elapsed_s"]:.1f}')
print(f'anchors: {d["config"]["anchors"]}')
print(f'seeds:   {d["config"]["seeds"]}')
print(f'modes:   {d["config"]["modes"]}')
print(f'windows: {d["config"]["windows"]}')
print(f'holdout_days: {d["config"]["holdout_days"]}  commission_bps: {d["config"]["commission_bps"]}')

[load] FALLBACK to embedded snapshot (JSON absent on disk)

timestamp: 20260806_165146  coin: XRP-USD  device: cuda
data_hash: 6e49685b9f9ab297  sweep_elapsed_s: 2862.5
anchors: ['2025-09-30', '2025-12-31', '2026-03-31', '2026-06-30']
seeds:   [0, 1, 7, 42]
modes:   ['fresh', 'aged-1q']
windows: ['sliding', 'expanding']
holdout_days: 90  commission_bps: 10


## 1. Tableau des verdicts §C par bucket

Doctrine §C appliquée par bucket (32 modèles effectifs sur 64 tentés) :

- **BEATS** = edge_pp > 0 AND edge_σ ≥ +2.0 AND dm_p_median < 0.05
- **NO-BEATS** = edge_pp < 0 OR (edge_pp > 0 AND edge_σ < +2.0)
- **INCONCLUSIVE** = edge_pp > 0 AND edge_σ ≥ +2.0 BUT dm_p_median ≥ 0.05

Note : `edge_pp` = mean(dt_net_sharpe - bh_sharpe) × 100. `edge_σ` = edge_pp / std
de la différence cross-seed (heuristique bootstrap sur 8 observations par bucket).

In [2]:
# Calcul du verdict par bucket selon la doctrine §C
def verdict_c(edge_pp, edge_sigma, dm_p_median):
    if edge_pp < 0:
        return 'NO-BEATS'
    if edge_sigma >= 2.0 and dm_p_median < 0.05:
        return 'BEATS'
    if edge_sigma >= 2.0 and dm_p_median >= 0.05:
        return 'INCONCLUSIVE (edge≥2σ but DM ns)'
    return 'NO-BEATS (edge<2σ)'

rows = []
for b in d['summary_by_bucket']:
    v = verdict_c(b['edge_mean_pp'], b['edge_sigma'], b['dm_p_median'])
    rows.append({
        'bucket': b['label'],
        'n': b['n_models'],
        'dt_sharpe': round(b['dt_net_sharpe_mean'], 3),
        'edge_pp': round(b['edge_mean_pp'], 2),
        'edge_σ': round(b['edge_sigma'], 2),
        'dm_p_med': round(b['dm_p_median'], 4),
        'retrain_s': round(b['mean_retrain_s'], 1),
        'verdict §C': v,
    })

import pandas as pd
df_verdict = pd.DataFrame(rows)
print(df_verdict.to_string(index=False))
print()

# Resume global
verdict_counts = df_verdict['verdict §C'].str.split(' ', n=1).str[0].value_counts()
print('=== VERDICT GLOBAL ===')
for v_label, cnt in verdict_counts.items():
    print(f'  {v_label:>10} : {cnt} bucket(s)')

           bucket  n  dt_sharpe  edge_pp  edge_σ  dm_p_med  retrain_s         verdict §C
    fresh/sliding  8      0.586   148.51    0.76    0.3090       49.8 NO-BEATS (edge<2σ)
  aged-1q/sliding  8      0.514   141.37    0.71    0.3272       47.7 NO-BEATS (edge<2σ)
  fresh/expanding  8      0.389   128.86    0.65    0.3090      135.8 NO-BEATS (edge<2σ)
aged-1q/expanding  8      0.481   138.01    0.74    0.3090      124.5 NO-BEATS (edge<2σ)

=== VERDICT GLOBAL ===
    NO-BEATS : 4 bucket(s)


## 2. SUSPECT cross-seed dup — signal de dégénérescence

Observation critique : sur l'anchor **2025-12-31** (train_end = fin Q4 2025), les
4 seeds en mode `fresh` produisent **EXACTEMENT** `dt_net_sharpe = +1.5593` au
bit près. Pareil pour `dt_gross_sharpe = +1.5636` et `bh_sharpe = -1.5636`.

Hypothèse : le modèle DT a dégénéré vers la stratégie triviale **"inverse exact
de buy-and-hold"** (short-BH systématique). Ce n'est PAS un vrai edge — c'est
une **stratégie de signal nul** qui produit l'inverse par défaut. La dégénérescence
se produit quand le modèle n'arrive plus à extraire de feature actionnable du
contexte et bascule sur un comportement réflexe.

**Critère de détection** : `dt_net_sharpe ≈ -bh_sharpe` (au bit près, sur ≥3 seeds)
pour UN anchor. Si vérifié → **finding = "modèle dégénéré sur ce régime"**, pas
"edge réel".

In [3]:
# Detection du SUSPECT cross-seed dup par (mode, window, anchor)
import collections

per_seed = d['per_seed_snapshot']

# Group par (bucket, train_end)
groups = collections.defaultdict(list)
for s in per_seed:
    key = (s['bucket'], s['train_end'])
    groups[key].append(s)

suspects = []
for (bucket, train_end), entries in groups.items():
    dt_values = [e['dt_net_sharpe'] for e in entries]
    bh_values = [e['bh_sharpe'] for e in entries]
    n_seeds = len(entries)
    # Critere : dt_net identique au bit pres (toutes valeurs egales)
    is_uniform_dt = len(set(round(v, 6) for v in dt_values)) == 1
    # Critere : dt_net = -bh_net (inverse exact)
    is_inverse = all(abs(dt + bh) < 1e-3 for dt, bh in zip(dt_values, bh_values))
    if is_uniform_dt or is_inverse:
        suspects.append({
            'bucket': bucket,
            'train_end': train_end,
            'n_seeds': n_seeds,
            'dt_value': dt_values[0],
            'bh_value': bh_values[0],
            'momentum_naked': entries[0]['momentum_naked_net_sharpe'],
            'uniform_dt': is_uniform_dt,
            'inverse_bh': is_inverse,
        })

print(f'=== SUSPECT CROSS-SEED DUP ({len(suspects)} groupes concernes) ===')
for s in suspects:
    flags = []
    if s['uniform_dt']:
        flags.append('UNIFORM')
    if s['inverse_bh']:
        flags.append('INVERSE_BH')
    flag_str = '+'.join(flags) if flags else '?'
    print(f"  {s['bucket']:>20} | train_end={s['train_end']} | "
          f"{s['n_seeds']} seeds | "
          f"dt={s['dt_value']:+.4f} bh={s['bh_value']:+.4f} | "
          f"mom={s['momentum_naked']:+.4f} | [{flag_str}]")

# Sanity : confirmer l'identite au bit
print()
print('=== Confirmation bit-pres sur fresh/sliding 2025-12-31 ===')
fresh_q4 = [e for e in per_seed if e['bucket'] == 'fresh/sliding' and e['train_end'] == '2025-12-31']
for e in fresh_q4:
    print(f'  seed={e["seed"]:>3} dt_net={e["dt_net_sharpe"]:.16f}')
print(f'Toutes valeurs identiques (delta max) : '
      f'{max(fresh_q4, key=lambda x: x["dt_net_sharpe"])["dt_net_sharpe"] - min(fresh_q4, key=lambda x: x["dt_net_sharpe"])["dt_net_sharpe"]:.2e}')

=== SUSPECT CROSS-SEED DUP (3 groupes concernes) ===
         fresh/sliding | train_end=2025-12-31 | 4 seeds | dt=+1.5593 bh=-1.5636 | mom=+0.2449 | [UNIFORM]
       fresh/expanding | train_end=2025-12-31 | 4 seeds | dt=+1.5593 bh=-1.5636 | mom=+0.2449 | [UNIFORM]
     aged-1q/expanding | train_end=2025-09-30 | 4 seeds | dt=+1.5593 bh=-1.5636 | mom=+0.2449 | [UNIFORM]

=== Confirmation bit-pres sur fresh/sliding 2025-12-31 ===
  seed=  0 dt_net=1.5592999999999999
  seed=  1 dt_net=1.5592999999999999
  seed=  7 dt_net=1.5592999999999999
  seed= 42 dt_net=1.5592999999999999
Toutes valeurs identiques (delta max) : 0.00e+00


## 3. Decay fresh → aged-1q — coût de ré-entraînement vs gain marginal

Question opérationnelle : si on **ré-entraîne tous les anchors** (mode `fresh`)
vs **ré-utilise un modèle âgé de 1 trimestre** (mode `aged-1q`), perd-on
significativement de l'edge ?

Métrique : `decay_pp_per_quarter` = (aged1q_edge - fresh_edge) en pp. **Négatif =
le modèle âgé perd de l'edge** (ré-entraînement utile). **Positif = le modèle
âgé est MEILLEUR que le fresh** (ré-entraînement inutile).

In [4]:
# Tableau decay fresh vs aged-1q + cout de re-entrainement
decay = d['decay_fresh_vs_aged1q']

print('=== DECAY (fresh vs aged-1q) ===')
print(f"{'window':<12} {'fresh_pp':>10} {'aged1q_pp':>11} {'decay_pp/Q':>12}")
print('-' * 50)
for w in ['sliding', 'expanding']:
    e = decay[w]
    print(f'{w:<12} {e["fresh_edge_pp"]:>+10.2f} {e["aged1q_edge_pp"]:>+11.2f} '
          f'{e["decay_pp_per_quarter"]:>+12.2f}')
print()

# Cout de re-entrainement (s/seed) par bucket
print('=== Cout de re-entrainement (mean_retrain_s par bucket) ===')
print(f"{'bucket':<20} {'retrain_s':>10}")
print('-' * 32)
for b in d['summary_by_bucket']:
    print(f"{b['label']:<20} {b['mean_retrain_s']:>10.1f}")
print()

# Verdict operationnel :
# - sliding : decay +7.14 pp/Q (fresh +148.51 -> aged1q +141.37). Fresh est 7.14pp
#   MIEUX que aged1q. Cout : ~50s/anchor pour gagner 7.14pp/Q.
# - expanding : decay -9.15 pp/Q (fresh +128.86 -> aged1q +138.01). Aged1q est
#   9.15pp MIEUX que fresh. Re-entrainement inutile (ou contre-productif).
print('=== Verdict operationnel ===')
sliding_decay = decay['sliding']['decay_pp_per_quarter']
expanding_decay = decay['expanding']['decay_pp_per_quarter']
if sliding_decay > 0 and expanding_decay < 0:
    print(f"  sliding : fresh +{sliding_decay:.1f}pp > aged1q. Re-train JUSTIFIE (~50s/anchor).")
    print(f"  expanding : aged1q +{abs(expanding_decay):.1f}pp > fresh. Re-train INUTILE (~135s/anchor gaspille).")
    print(f"  Recommandation : utiliser aged-1q sur expanding (pas de retrain),")
    print(f"                  fresh sur sliding (retrain justifie).")
elif abs(sliding_decay) < 5 and abs(expanding_decay) < 5:
    print(f"  Decay < 5 pp/Q dans les 2 cas -> re-train n'apporterien de significatif.")

=== DECAY (fresh vs aged-1q) ===
window         fresh_pp   aged1q_pp   decay_pp/Q
--------------------------------------------------
sliding         +148.51     +141.37        +7.14
expanding       +128.86     +138.01        -9.15

=== Cout de re-entrainement (mean_retrain_s par bucket) ===
bucket                retrain_s
--------------------------------
fresh/sliding              49.8
aged-1q/sliding            47.7
fresh/expanding           135.8
aged-1q/expanding         124.5

=== Verdict operationnel ===
  sliding : fresh +7.1pp > aged1q. Re-train JUSTIFIE (~50s/anchor).
  expanding : aged1q +9.2pp > fresh. Re-train INUTILE (~135s/anchor gaspille).
  Recommandation : utiliser aged-1q sur expanding (pas de retrain),
                  fresh sur sliding (retrain justifie).


## 4. Vs momentum naked — le vrai adversaire, pas BH

Le gain vs buy-and-hold (+128 à +148 pp cross-bucket) est trompeur : il vient
principalement de **momentum naked qui bat BH sur ce régime**. La question
pertinente est : **est-ce que DT apporte un edge AU-DESSUS de momentum naked ?**

Un edge positif vs momentum naked justifierait la complexité du Decision
Transformer (architecture lourde, retraining coûteux). Un edge négatif = DT
**n'apporte rien** par-dessus une simple règle momentum.

In [5]:
# Edge vs momentum naked par bucket
import statistics as st

print('=== EDGE DT vs MOMENTUM NAKED (par bucket) ===')
print(f"{'bucket':<20} {'dt_sharpe':>10} {'mom_sharpe':>12} {'edge_sharpe':>13}")
print('-' * 60)
edge_vs_mom_per_bucket = {}
for bucket_label in ['fresh/sliding', 'aged-1q/sliding', 'fresh/expanding', 'aged-1q/expanding']:
    entries = [e for e in per_seed if e['bucket'] == bucket_label]
    dt_mean = st.mean(e['dt_net_sharpe'] for e in entries)
    mom_mean = st.mean(e['momentum_naked_net_sharpe'] for e in entries)
    edge_sharpe = dt_mean - mom_mean
    edge_vs_mom_per_bucket[bucket_label] = edge_sharpe
    print(f'{bucket_label:<20} {dt_mean:>+10.3f} {mom_mean:>+12.3f} {edge_sharpe:>+13.3f}')

print()
print('=== Verdict vs momentum naked ===')
all_negative = all(v < 0 for v in edge_vs_mom_per_bucket.values())
print(f"  DT perd vs momentum naked sur TOUS les buckets : {all_negative}")
print(f"  Edge Sharpe (DT - mom) : "
      f"min={min(edge_vs_mom_per_bucket.values()):+.3f} / "
      f"max={max(edge_vs_mom_per_bucket.values()):+.3f}")
print()
print('Conclusion : le gain vs BH observe est illusoire -- il vient de momentum')
print('naked qui bat BH, pas de DT. Decision Transformer = zero edge marginal.')
print('Recommandation : ne PAS deployer DT en fold-wise. Utiliser momentum naked')
print('(regle directionnelle triviale, pas de GPU ni de training requis).')

=== EDGE DT vs MOMENTUM NAKED (par bucket) ===
bucket                dt_sharpe   mom_sharpe   edge_sharpe
------------------------------------------------------------
fresh/sliding            +0.586       +0.777        -0.191
aged-1q/sliding          +0.515       +0.777        -0.262
fresh/expanding          +0.389       +0.777        -0.387
aged-1q/expanding        +0.481       +0.777        -0.296

=== Verdict vs momentum naked ===
  DT perd vs momentum naked sur TOUS les buckets : True
  Edge Sharpe (DT - mom) : min=-0.387 / max=-0.191

Conclusion : le gain vs BH observe est illusoire -- il vient de momentum
naked qui bat BH, pas de DT. Decision Transformer = zero edge marginal.
Recommandation : ne PAS deployer DT en fold-wise. Utiliser momentum naked
(regle directionnelle triviale, pas de GPU ni de training requis).


## 5. Visualisation — edge_σ vs seuil §C + decay slope

Deux plots :
1. **Edge_σ par bucket** avec ligne horizontale au seuil §C (edge_σ = 2.0).
   Aucun bucket ne franchit le seuil → visualisation directe du NO-BEATS.
2. **Decay slope** fresh vs aged-1q par window : sliding (+) vs expanding (-)
   = régimes opposés.

In [6]:
# Plots : edge_σ par bucket + decay slope
import matplotlib
matplotlib.use('Agg')  # pas d'affichage interactif en Papermill CI
import matplotlib.pyplot as plt

# Plot 1 : edge_sigma par bucket + seuil §C
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

buckets = [b['label'] for b in d['summary_by_bucket']]
edge_sigmas = [b['edge_sigma'] for b in d['summary_by_bucket']]
colors = ['tab:red' if s < 2.0 else 'tab:green' for s in edge_sigmas]

ax1.barh(buckets, edge_sigmas, color=colors, edgecolor='black', linewidth=0.5)
ax1.axvline(x=2.0, color='tab:green', linestyle='--', linewidth=1.5, label='Seuil §C BEATS (2σ)')
ax1.axvline(x=0.0, color='tab:gray', linestyle='-', linewidth=0.5)
ax1.set_xlabel('edge_σ (DT net vs BH)')
ax1.set_title('c.1330 — Edge_σ par bucket vs seuil §C = 2.0')
ax1.legend(loc='lower right')
ax1.grid(axis='x', alpha=0.3)
for i, v in enumerate(edge_sigmas):
    ax1.text(v + 0.05, i, f'{v:+.2f}', va='center', fontsize=9)

# Plot 2 : decay fresh vs aged-1q par window
windows = ['sliding', 'expanding']
fresh_pp = [d['decay_fresh_vs_aged1q'][w]['fresh_edge_pp'] for w in windows]
aged_pp = [d['decay_fresh_vs_aged1q'][w]['aged1q_edge_pp'] for w in windows]

x = range(len(windows))
w_bar = 0.35
ax2.bar([i - w_bar/2 for i in x], fresh_pp, w_bar, label='fresh',
        color='tab:blue', edgecolor='black', linewidth=0.5)
ax2.bar([i + w_bar/2 for i in x], aged_pp, w_bar, label='aged-1q',
        color='tab:orange', edgecolor='black', linewidth=0.5)
ax2.set_xticks(list(x))
ax2.set_xticklabels(windows)
ax2.set_ylabel('Edge (pp)')
ax2.set_title('c.1330 — Decay fresh vs aged-1q par window')
ax2.axhline(y=0, color='tab:gray', linestyle='-', linewidth=0.5)
ax2.legend()
ax2.grid(axis='y', alpha=0.3)
for i, (f, a) in enumerate(zip(fresh_pp, aged_pp)):
    ax2.text(i - w_bar/2, f + 2, f'{f:+.1f}', ha='center', fontsize=9)
    ax2.text(i + w_bar/2, a + 2, f'{a:+.1f}', ha='center', fontsize=9)

plt.tight_layout()
out_path = Path('c1330_xrp_dt_foldwise_plots.png')
plt.savefig(out_path, dpi=110, bbox_inches='tight')
print(f'Plot sauvegarde : {out_path} ({out_path.stat().st_size} bytes)')
plt.close()

Plot sauvegarde : c1330_xrp_dt_foldwise_plots.png (53500 bytes)


Sur l'anchor fin-Q4-2025 (et marginalement fin-Q3 en aged-1q/expanding), **tous
les modèles DT convergent au bit près vers `dt_net ≈ -bh_sharpe`** (stratégie
"inverse exact de BH"). **3 groupes (bucket × train_end) × 4 seeds = 12 entrées**
sont dégénérées de la même manière, soit ~37.5% des 32 modèles effectifs. C'est
**un signal de collapse de feature** : le DT n'extrait plus rien d'actionnable
et bascule sur l'inverse systématique.